# Wikipedia knowledge graph + Text2Cypher — interactive tour

The **LangChain-forward** demo. An LLM (`LLMGraphTransformer`) built this graph
from raw Wikipedia article leads; natural-language questions are answered by
**generating Cypher** through an LCEL pipeline.

**Prerequisite — build the graph first:**

```bash
cd langchain
WIKI_LIMIT=200 .venv/bin/python examples/demos/02_wikipedia_kg/build_kg.py
```

In [1]:
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../02_wikipedia_kg
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for `_common`
sys.path.insert(0, str(HERE))              # this dir, for `ask`

import pandas as pd
from _common import agens, config
from _common.models import get_llm
import ask                                  # the Text2Cypher LCEL chain lives in ask.py

GRAPH = "wikipedia_kg"
# enhanced_schema=True -> the schema carries example property values, which the
# LLM uses to write better Cypher.
graph = agens.make_graph(GRAPH, create=False, enhanced_schema=True)
print("connected to", config.url().split("@")[-1])

connected to localhost:55432/agensgraph_demos


## The knowledge graph the LLM built

Typed entities + LLM-named relationships, plus `Document` provenance nodes
(`(:Document)-[:MENTIONS]->(entity)`).

In [2]:
n = graph.query("MATCH (n) RETURN count(n) AS c")[0]["c"]
e = graph.query("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"]
print(f"{n:,} nodes · {e:,} edges")
pd.DataFrame(graph.query(
    "MATCH (n) RETURN label(n) AS label, count(*) AS count ORDER BY count DESC"
))

2,860 nodes · 4,856 edges


,label,count
0,Concept,630
1,Person,567
2,Location,464
3,Work,283
4,Organization,272
5,Group,170
6,Event,145
7,Document,134
8,Field,131
9,Technology,43


In [3]:
# A peek at some extracted relationships (excluding the Document->entity
# provenance edges, so we see entity-to-entity relationships).
pd.DataFrame(graph.query(
    "MATCH (a)-[r]->(b) WHERE type(r) <> 'MENTIONS' "
    "RETURN label(a) AS from_type, a.id AS source, type(r) AS rel, b.id AS target LIMIT 12"
))

,from_type,source,rel,target
0,Person,Achilles,NAMED_AFTER,Achilles tendon
1,Concept,Animation,USES,Computer-Generated Imagery
2,Concept,Apollo,ATTRIBUTE_OF,lyre
3,Concept,Assistive technology,EXAMPLE_OF,wheelchairs
4,Concept,Assistive technology,EXAMPLE_OF,assistive eating devices
5,Concept,Assistive technology,EXAMPLE_OF,voice recognition software
6,Technology,abacus,REPLACED_BY,calculators
7,Technology,abacus,REPLACED_BY,computers
8,Concept,Bitumen,REFINED_FROM,crude oils
9,Event,Apollo 8,USED,Saturn V


## What the Text2Cypher model sees

The graph's schema (labels, properties, example values, relationship patterns) is
fed to the LLM so it writes valid AgensGraph Cypher.

In [4]:
print(graph.get_schema.strip()[:1500])

Node properties are the following:
        [{'labels': 'Award', 'properties': [{'type': 'STRING', 'property': 'id', 'examples': ['Academy Award for Best Production Design', 'BAFTA Award for Best Production Design', "Critics' Choice Movie Award for Best Production Design"]}]}, {'labels': 'Concept', 'properties': [{'type': 'STRING', 'property': 'id', 'examples': ['Anarchism', 'libertarian socialism', 'anti-capitalist movement']}]}, {'labels': 'Document', 'properties': [{'type': 'STRING', 'property': 'id', 'examples': ['3217bea3396440f7861e71ca5cc463bc', '7ac2c48e1665150e83d8c057640d598b', '7a59fef7cb4cb06acf925740e6069d95']}, {'type': 'STRING', 'property': 'source', 'examples': ['wikipedia']}, {'type': 'STRING', 'property': 'title', 'examples': ['Anarchism', 'Albedo', 'Alabama']}, {'type': 'STRING', 'property': 'url', 'examples': ['https://en.wikipedia.org/wiki/Anarchism', 'https://en.wikipedia.org/wiki/Albedo', 'https://en.wikipedia.org/wiki/Alabama']}]}, {'labels': 'Event', 'properties

## Ask in natural language (LCEL Text2Cypher)

`ask.build_chain` composes the pipeline with LangChain Expression Language:

```python
RunnablePassthrough.assign(cypher = cypher_prompt | llm | StrOutputParser() | clean)
| RunnablePassthrough.assign(results = run_cypher)          # read-only, timed
| RunnablePassthrough.assign(answer  = answer_prompt | llm | StrOutputParser())
```

In [5]:
chain = ask.build_chain(graph, get_llm())
schema = graph.get_schema

def run(question):
    out = chain.invoke({"schema": schema, "question": question})
    print("Q:", question)
    print("\nGenerated Cypher:\n  " + out["cypher"].replace("\n", "\n  "))
    print("\nAnswer:\n" + out["answer"])

run("What types of entities are in the graph, and how many of each?")

Q: What types of entities are in the graph, and how many of each?

Generated Cypher:
  MATCH (n) RETURN label(n) AS type, count(*) AS n ORDER BY n DESC LIMIT 50

Answer:
The types of entities in the graph and their counts are as follows:

- Concept: 630
- Person: 567
- Location: 464
- Work: 283
- Organization: 272
- Group: 170
- Event: 145
- Document: 134
- Field: 131
- Technology: 43
- Award: 21


In [6]:
run("Which 5 people are connected to the most other entities?")

Q: Which 5 people are connected to the most other entities?

Generated Cypher:
  MATCH (p:"Person")-[r]->(e) RETURN p.id AS person, count(e) AS connections ORDER BY connections DESC LIMIT 5

Answer:
The 5 people connected to the most other entities are:
1. Alfons Maria Jakob - 24 connections
2. Albert Camus - 22 connections
3. August William Derleth - 20 connections
4. Andrei Tarkovsky - 20 connections
5. Alain Connes - 17 connections


In [7]:
run("List 5 organizations in the graph and one entity each is connected to.")

Q: List 5 organizations in the graph and one entity each is connected to.

Generated Cypher:
  MATCH (o:"Organization")-[r]->(e) RETURN o.id AS organization, e.id AS connected_entity LIMIT 5

Answer:
1. Confederate States of America - United States
2. Art Directors' branch of the Academy of Motion Picture Arts and Sciences - Designers' branch
3. The Guardian - Actrius
4. The Evening Standard - Actrius
5. University of Copenhagen - Bartholins


## What you can do with this

- **Build a graph from unstructured text** with one LangChain component
  (`LLMGraphTransformer`) — typed entities + relationships, structured output.
- **Query it in natural language** — the LLM writes schema-grounded, read-only
  Cypher; results ground the final answer. All composed with LCEL.

Try your own:

```python
run("your question here")
```

When finished, close the shared pool: `agens.close()`